In [2]:
import copy
import numbers
from sympy import *

In [2]:
def hrs_min_sec(sec_val):
    hours=str(sec_val//(60**2))
    minutes=str((sec_val//60)%60)
    seconds=str(round(sec_val%60,0))
    if sec_val//(60**2)!=0:
        return(hours+' hrs '+minutes+' min '+seconds+' sec')
    if (sec_val//60)%60!=0:
        return(minutes+' min '+seconds+' sec')
    return(seconds+' sec')

In [3]:
def find_cochain_basis(ss):
    '''args: ss (spanning set), a list of cochains of the same homogeneous degree
       Returns: A list of cochains which are a basis for the subspace spanned by ss'''
    if len(ss)==0: return []
    basis_set=set()
    for c in ss:
        for base_elt in c.coeff_dict:
            basis_set.add(base_elt)
    B=list(basis_set)
    
    M=zeros(len(ss),len(B))
    for i in range(len(ss)):
        set_row(M,i,coordinatize_cochain_in_basis(ss[i],B))
    M=M.rref()[0]
    result=[list(M.row(i)) for i in range(shape(M)[0]) 
            if list(M.row(i))!=[0]*len(M.row(i))]   
    basis_cochains=[cochain({A:1},ss[0].parent) for A in B]
    return([coords_to_lin_comb(A,basis_cochains) for A in result])

In [1]:
def simplify_cochain(c):
    # How can I make this faster?
    ctr=0
#     print('Total coeffs:',len(list(c.coeff_dict.keys())))
    for k in c.coeff_dict:
#         print(ctr,'coeffs simplified; prev. time', round(time.time()-time0,2))
        c.coeff_dict[k]=simplify(c.coeff_dict[k])
        ctr+=1
    c.remove_zeros()

In [4]:
def set_row(mat,rowNum,row):
    if type(row)==type(zeros(3,3)):
        rowList=list(row)
    else: 
        if type(row)==type([0]):
            rowList=row
        else: print('setRow error: arg row must be either matrix or list')
    if len(rowList)!=len(mat.row(0)):
        print('setRow error: mat.row() and row have differing lengths')
        return None
    for i in range(len(rowList)):
        mat[rowNum,i]=rowList[i]
        
def set_col(mat,colNum,col):
    if type(col)==type(zeros(2,2)):
        colList=list(col)
    else:
        if type(col)==type([0]):
            colList=col
        else: print('SetCol error: arg col must be either matrix or list')
            
    if len(colList)!=len(mat.col(0)):
        print('SetCol error: mat.col() and col have differing lengths')
        return None
    for i in range(len(colList)):
        mat[i,colNum]=colList[i]

In [5]:
def coords_to_lin_comb(basis,coords):
    '''args: basis, a list of symbols, and coords, a vector of the same length
       Returns: A LinComb corresponding to the vector coords
       NOTE: basis must be a list of symbols'''
    
    # Check if the lengths are the same:
    if len(basis)!=len(coords):
        print('coords_to_lin_comb error: |basis| and |coords| have different lengths')
        return None
    
    result=0
    for i in range(len(basis)):
        result=result+coords[i]*basis[i]
    return result

In [6]:
def coordinatize_cochain_in_basis(c,basis):
    '''c: a cochain object of homogeneous degree
       basis: a collection of str tuples representing elementary cochains
       returns the vector representation of c w.r.t basis as a list'''
    result=[0]*len(basis)
    for key in c.coeff_dict:
        if key in basis: result[basis.index(key)]=c.coeff_dict[key]
        else: print('coordinatize_cochain_in_basis error: cochain component',key,'not in basis')
    return result

In [7]:
def convert_T_symb_elt_to_cochain(se):
    '''se: a T_symb_elt object or a T_symb_basis object
       returns: a degree 0 cochain object corresponding to se'''
    if se==0: return cochain({},se.heis_dim)
    return cochain({(str(T_symb_basis[i]),):se.vec_rep[i] 
                    for i in range(len(T_symb_basis))},se.heis_dim)

In [8]:
def permutation_sign(it_1,it_2):
    '''tuple_1, tuple_2: iterables containing the same elements
       returns: the sign of the permutation taking it_1 to it_2'''
    cnt=0
    for i in range(len(it_1)):
        for j in range(i+1,len(it_1)):
            if it_2.index(it_1[j])<it_2.index(it_1[i]):
                cnt+=1
    return (-1)**cnt

In [9]:
def remove_antisymm_zeros(coeff_dict):
    '''coeff_dict: a coeff_dict for an exterior vector or cochain
       result: coeff_dict, but with keys like (e1,e2,e1) removed'''
    result_dict=copy.copy(coeff_dict)
    for key in list(result_dict):
        if len(set(key))!=len(key):
            result_dict.pop(key)
    return result_dict

In [10]:
def remove_antisymm_zeros_cochains(coeff_dict):
    '''coeff_dict: a coeff_dict for an exterior vector or cochain
       result: coeff_dict, but with keys like (e1,e1,e2) removed,
       leaving keys like (e1,e2,e1) representing nontrivial cochains'''
    
    result_dict=copy.copy(coeff_dict)
    for key in list(result_dict):
        if len(set(key[0:len(key)-1]))!=len(key)-1:
            result_dict.pop(key)
    return result_dict

In [11]:
def merge_coeff_dicts(dict1,dict2):
    result={}
    for key in set(dict1.keys()).union(set(dict2.keys())):
        coeff=0
        if key in dict1:
            coeff+=dict1[key]
        if key in dict2:
            coeff+=dict2[key]
        result[key]=coeff
    return remove_zeros(result)

In [12]:
def remove_zeros(coeff_dict):
    '''coeff_dict: a dict with integer values
       returns: a copy of coeff_dict with all keys of value 0 removed'''
    return{A:coeff_dict[A] for A in coeff_dict if coeff_dict[A]!=0}

In [13]:
def str_from_coeff_dict(coeff_dict):
    '''coeff_dict: a dict with keys that are printable objects
                   or tuples of printable objects and integer values
       returns: a string representing the dict'''
    coeff_dict=remove_zeros(coeff_dict)
    if coeff_dict=={}: return '0'
    key_list=list(coeff_dict.keys())
    result=''

    for key in key_list:
        if type(key)==tuple:
            key_str='('+','.join([str(A) for A in key])+')'
        else: key_str=str(key)
        if coeff_dict[key]==1:
            result+=' + '+key_str
        elif coeff_dict[key]==-1:
            result+=' - '+key_str
        elif coeff_dict[key]!=0:
            # Does coeff_str need parens?
            if type(coeff_dict[key])==Add:
                coeff_str='('+str(coeff_dict[key])+')'
            else:
                coeff_str=str(coeff_dict[key])
            result+=' + '+coeff_str+'*'+key_str
    if result[0:3]==' + ':
        return result[3:len(result)]
    return result[1:len(result)]

In [14]:
def iprod_mat(elts):
    '''elts: an iterable of elements with attribute iprod
       returns: the matrix with (ei.iprod(ej)) as its (i,j)-entry'''
    result=zeros(len(elts))            
    for i in range(len(elts)):
        c1=elts[i]
        for j in range(i,len(elts)):
            c2=elts[j]
            val=c1.iprod(c2)
            result[i,j]=val
            result[j,i]=val
    return result

In [15]:
def augment_with_identity(M):
    '''M: a matrix
       returns: matrix (M|Id(n)), where n=shape(M)[0], the number of columns'''
    result=M
    for i in range(shape(M)[0]):
        new_col=[0]*shape(M)[0]
        new_col[i]=1
        new_col=Matrix([[k] for k in new_col])
        result=result.col_insert(shape(M)[1]+i,new_col)
    return result

In [16]:
def col_sp_and_preim(M):
    '''arg: a Matrix M representing a f.d. lin. op.
       returns: a pair of lists of coords (A,B) where
                A = elements of the domain which map to the elts of B
                B = basis for the columnspace of M'''
    result=([],[])
    
    # transpose and augment
    T=transpose(M)
    T=augment_with_identity(T)
    ## row reduce
    red=T.rref()
    T=red[0]

    piv=red[1]
    i=0
    while piv[i]<shape(M)[0]:
        result[0].append(T.row(i)[shape(M)[0]:len(T.row(i))])
        result[1].append(T.row(i)[0:shape(M)[0]])
        i+=1
        if i==len(piv): break
    return result

In [17]:
def augment_with_identity(M):
    '''M: a matrix
       returns: matrix (M|Id(n)), where n=shape(M)[0], the number of columns'''
    result=M
    for i in range(shape(M)[0]):
        new_col=[0]*shape(M)[0]
        new_col[i]=1
        new_col=Matrix([[k] for k in new_col])
        result=result.col_insert(shape(M)[1]+i,new_col)
    return result

In [18]:
def perm_sign(L,sort_L):
    '''args: two lists with the same elements;
       returns: the sign of the permutation (slow)'''
    result=1
    for j in range(len(L)):
        pj=sort_L.index(L[j])
        for i in range(j):
            if pj<sort_L.index(L[i]): result=result*(-1)
    return result

In [19]:
def ind_der(ind_expr,i):
    '''args: ind_expr, an expression in coordinates h,e,y, and indexed objects, and a natural number i
       returns: the normal form of the derivative of the expression in the i direction'''
    # I'm not sure if this simplification will help or hurt time efficiency
    ind_expr=simplify(ind_expr)
    if isinstance(ind_expr,numbers.Number):
        return 0
    if type(ind_expr)==Add:
        result = Add(*[abn_ind_der(A,i) for A in ind_expr.args])
        return normal_form(result)
    if type(ind_expr)==Mul:
        result=0
        for j in range(len(ind_expr.args)):
            result+=abn_ind_der(ind_expr.args[j],i)*Mul(*list(ind_expr.args[0:j]+ind_expr.args[j+1:len(ind_expr.args)]))
        return normal_form(result)
    # Here I assume the exponents are constant
    if type(ind_expr)==Pow:
        return normal_form(ind_expr.exp*ind_expr.base**(ind_expr.exp-1)*abn_ind_der(ind_expr.base,i))
    if type(ind_expr)==Indexed:
        base=ind_expr.base
        ind=list(ind_expr.indices)
        return normal_form(base[ind+[i]])
    if type(ind_expr)==Symbol:
        if ind_expr in [y,h,e] and [y,h,e].index(ind_expr)==i: return 1
        else: return 0
    
def abn_ind_der(ind_expr,i):
    '''args: ind_expr, an expression in coordinates h,e,y, and indexed objects, and a natural number i
       returns: the derivative of the expression in the i direction without reducing to normal form'''
    # I'm not sure if this simplification will help or hurt time efficiency
    h,e,y = symbols('h,e,y')
    ind_expr=simplify(ind_expr)
    if isinstance(ind_expr,numbers.Number):
        return 0
    if type(ind_expr)==Add:
        result = Add(*[abn_ind_der(A,i) for A in ind_expr.args])
        return result
    if type(ind_expr)==Mul:
        result=0
        for j in range(len(ind_expr.args)):
            result+=abn_ind_der(ind_expr.args[j],i)*Mul(*list(ind_expr.args[0:j]+ind_expr.args[j+1:len(ind_expr.args)]))
        return result
    if type(ind_expr)==Pow:
        return ind_expr.exp*ind_expr.base**(ind_expr.exp-1)*abn_ind_der(ind_expr.base,i)
    if type(ind_expr)==Indexed:
        base=ind_expr.base
        ind=list(ind_expr.indices)
        return base[ind+[i]]
    if type(ind_expr)==Symbol:
        if ind_expr in [y,h,e] and [y,h,e].index(ind_expr)==i: return 1
        else: return 0
    if type(ind_expr)==exp:
        return ind_expr*abn_ind_der(ind_expr.args[0],i)

In [52]:
def reduce_numer(expr):
    '''arg: expr, a rational expression in symbolic variables 
       returns: the simplest form of the numerator of expr'''
    return numer(simplify(ratsimp(expr)))
    

In [ ]:
def Indexed_factors(expr):
    '''args: expr, a polynomial in Indexed objects
       returns: a set of the Indexed objects involved in expr'''
    if type(expr)==Pow:
        return Indexed_factors(expr.as_base_exp()[0])
    if type(expr)==Mul:
        return set.union(*[Indexed_factors(A) for A in expr.as_coeff_mul()[1]])
    if type(expr)==Add:
        return set.union(*[Indexed_factors(A) for A in expr.as_coeff_add()[1]])
    if type(expr)==Indexed:
        return set([expr])
    return set()

### reduce_numer test

In [48]:
# K=IndexedBase('K')

In [49]:
# P1=((K[0]*K[1])/(K[2]+K[3])+K[1]**2-7*K[2])/(K[8]+K[2])-(4*K[3]/5*K[1]+(K[4]**3-K[1]*K[2])/(K[5]**4+1)/(
#     -3*K[2]**2+K[1])-K[3]/(K[5]+K[1]/K[6]))/(K[4]**2-7*K[1]+3/K[4])


In [50]:
# display(P1)

-((-K[1]*K[2] + K[4]**3)/((K[1] - 3*K[2]**2)*(K[5]**4 + 1)) + 4*K[1]*K[3]/5 - K[3]/(K[1]/K[6] + K[5]))/(-7*K[1] + K[4]**2 + 3/K[4]) + (K[1]**2 - 7*K[2] + K[0]*K[1]/(K[2] + K[3]))/(K[2] + K[8])

In [51]:
# reduce_numer(P1)

7*K[0]*K[1]**4*K[4]*K[5]**4 + 7*K[0]*K[1]**4*K[4] - 21*K[0]*K[1]**3*K[2]**2*K[4]*K[5]**4 - 21*K[0]*K[1]**3*K[2]**2*K[4] - K[0]*K[1]**3*K[4]**3*K[5]**4 - K[0]*K[1]**3*K[4]**3 + 7*K[0]*K[1]**3*K[4]*K[5]**5*K[6] + 7*K[0]*K[1]**3*K[4]*K[5]*K[6] - 3*K[0]*K[1]**3*K[5]**4 - 3*K[0]*K[1]**3 + 3*K[0]*K[1]**2*K[2]**2*K[4]**3*K[5]**4 + 3*K[0]*K[1]**2*K[2]**2*K[4]**3 - 21*K[0]*K[1]**2*K[2]**2*K[4]*K[5]**5*K[6] - 21*K[0]*K[1]**2*K[2]**2*K[4]*K[5]*K[6] + 9*K[0]*K[1]**2*K[2]**2*K[5]**4 + 9*K[0]*K[1]**2*K[2]**2 - K[0]*K[1]**2*K[4]**3*K[5]**5*K[6] - K[0]*K[1]**2*K[4]**3*K[5]*K[6] - 3*K[0]*K[1]**2*K[5]**5*K[6] - 3*K[0]*K[1]**2*K[5]*K[6] + 3*K[0]*K[1]*K[2]**2*K[4]**3*K[5]**5*K[6] + 3*K[0]*K[1]*K[2]**2*K[4]**3*K[5]*K[6] + 9*K[0]*K[1]*K[2]**2*K[5]**5*K[6] + 9*K[0]*K[1]*K[2]**2*K[5]*K[6] + 7*K[1]**5*K[2]*K[4]*K[5]**4 + 7*K[1]**5*K[2]*K[4] + 7*K[1]**5*K[3]*K[4]*K[5]**4 + 7*K[1]**5*K[3]*K[4] - 21*K[1]**4*K[2]**3*K[4]*K[5]**4 - 21*K[1]**4*K[2]**3*K[4] - 21*K[1]**4*K[2]**2*K[3]*K[4]*K[5]**4 - 21*K[1]**4*K[2]**2*

Each expression involving the structure functions $K$ can be written in many ways because of the Jacobi identity. We can write such expressions uniquely by requiring the lower indices to be ordered as well as the derivatives. Writing $K_{i,j,s}^k$ for $\mathcal{L}_sK^k_{i,j}$, we can apply commutation law

$$\mathcal{L}_s\mathcal{L}_tK = \mathcal{L}_{[s,t]}K + \mathcal{L}_t\mathcal{L}_sK = K_{s,t}^r\mathcal{L}_{r}K + \mathcal{L}_t\mathcal{L}_sK$$

to order derivatives and antisymmetry to order the lower indices.

In [ ]:
NF_dict={}

In [25]:
# Note: This uses reference to m, which isn't defined until Geometric Tanaka Prolongations is run

def normal_form(ind_expr,m):
    '''arg: ind_expr, a polynomial of indexed objects which each 
            represent structure functions for the symplectified distribution
            (ex: K[1,3,3,8,9] is the 8,9 derivative of 2-cochain K[1,3,3])
       returns: The normal form of the expression, exchanging derivatives as needed'''
    if ind_expr in NF_dict:
        return NF_dict[ind_expr]
    if type(ind_expr)==Add:
        result = simplify(Add(*[normal_form(A,m) for A in ind_expr.args]))
        NF_dict[ind_expr] = result
        return result
    if type(ind_expr)==Mul:
        result=simplify(Mul(*[normal_form(A,m) for A in ind_expr.args]))
        NF_dict[ind_expr] = result
        return result
    if isinstance(ind_expr,numbers.Number):
        NF_dict[ind_expr]=ind_expr
        return ind_expr
    if type(ind_expr)==Symbol:
        return ind_expr
    if type(ind_expr)==Pow:
        return normal_form(ind_expr.base,m)**normal_form(ind_expr.exp,m)
    if type(ind_expr)==Indexed:
        base=ind_expr.base
        deg=2
        im_ind=ind_expr.indices[2:3]
        ind=ind_expr.indices[0:2]
        ders=ind_expr.indices[3:len(ind_expr.indices)]
        
        if list(ind)!=sorted(ind):
            new_ind=sorted(ind)
            sgn=perm_sign(ind,new_ind)
            L=list(new_ind)+list(im_ind)+list(ders)
            result=simplify(normal_form(sgn*base[L],m))
            NF_dict[ind_expr]=result
            return result
        
        # Only 3 indices, in correct ordering
        if len(ind_expr.indices)==3:
            NF_dict[ind_expr]=ind_expr
            return ind_expr
        
        # More indices --> derivatives to check
        if list(ders)==sorted(ders):
            NF_dict[ind_expr]=ind_expr
            return ind_expr
        j=0 # Least int so that der[j]>der[j+1]
        while ders[j]<=ders[j+1]:
            j+=1
        
        result=base[list(im_ind)+list(ind)+list(ders[0:j])+[ders[j+1],ders[j]]]
        for l in range(2*m+5):
            result=result+base[ders[j],ders[j+1],l]*base[list(ind)+list(im_ind)+list(ders[0:j])+[l]]
        for i in ders[j+2:len(ders)]:
            result=abn_ind_der(result,i)
        result=normal_form(result,m)
        NF_dict[ind_expr]=result
        return result

In [20]:
# from sympy import *
# # An ad hoc test for col_sp_and_preim
# for m in range(1,10):
#     for n in range(1,10):
#         for qqqq in range(1,10):
#             A=randMatrix(m,n)
#             result=col_sp_and_preim(A)
#             r0=result[0]
#             r1=result[1]
#             for i in range(len(r0)):
#                 v= Matrix([[k] for k in r0[i]])
#                 test1=A*v
#                 if tuple(test1)!=r1[i]:
#                     print('Failure for A =', A)

In [21]:
# def coordinatize(b_mat,vec):
#     '''args: b_mat, a matrix such that the span of its columns include vec, a list
#        returns: a list coorinatizing vec in the basis b_mat'''
    

In [98]:
import itertools
def s(n):
    comb_list=itertools.combinations(range(0,len(a)), n)
    return sum([prod([a[i] for i in comb]) for comb in comb_list])